In [1]:
# mnist --> 5 layers(conv and pooling) --> 50 epochs

In [2]:
import numpy as np
from numba import cuda
import math

In [3]:
import torchvision
import torchvision.transforms as transforms

# Download MNIST
transform = transforms.ToTensor()

train_dataset = torchvision.datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

print("MNIST downloaded successfully")

100%|██████████| 9.91M/9.91M [00:04<00:00, 2.46MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 89.0kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 938kB/s] 
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.38MB/s]

MNIST downloaded successfully


In [4]:
image, label = train_dataset[0]

# Convert to numpy
image = image.squeeze().numpy().astype("float32")

print("Label:", label)
print("Shape:", image.shape)

Label: 5
Shape: (28, 28)


In [5]:
@cuda.jit
def conv2d(input_img, kernel, output):
    h, w = cuda.grid(2)

    H = input_img.shape[0]
    W = input_img.shape[1]

    if h < H and w < W:
        sum_val = 0.0

        for kh in range(3):
            for kw in range(3):
                ih = h + kh - 1
                iw = w + kw - 1

                if 0 <= ih < H and 0 <= iw < W:
                    sum_val += input_img[ih, iw] * kernel[kh, kw]

        output[h, w] = sum_val

In [6]:
@cuda.jit
def relu(x):
    i, j = cuda.grid(2)

    if i < x.shape[0] and j < x.shape[1]:
        if x[i, j] < 0:
            x[i, j] = 0

In [7]:
@cuda.jit
def maxpool_kernel(input_img, output):
    h, w = cuda.grid(2)

    H_out = output.shape[0]
    W_out = output.shape[1]

    if h < H_out and w < W_out:
        max_val = -1e9

        for i in range(2):
            for j in range(2):
                val = input_img[h*2 + i, w*2 + j]
                if val > max_val:
                    max_val = val

        output[h, w] = max_val

In [8]:
d_img = cuda.to_device(image)

In [9]:
kernels = []
for _ in range(5):
    k = np.random.randn(3,3).astype(np.float32) * 0.1
    kernels.append(cuda.to_device(k))

In [10]:
def launch_2d(kernel_func, input_arr, *args):
    H, W = input_arr.shape
    threads = (16,16)
    blocks = (math.ceil(H/16), math.ceil(W/16))
    kernel_func[blocks, threads](input_arr, *args)

In [11]:
# conv1
d_out1 = cuda.device_array((28,28), dtype=np.float32)
launch_2d(conv2d, d_img, kernels[0], d_out1)
launch_2d(relu, d_out1)

c:\Users\dil\AppData\Local\Programs\Python\Python313\Lib\site-packages\numba_cuda\numba\cuda\dispatcher.py:690: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


nvJitLinkError: ERROR_INTERNAL (6)
Linker error log: ERROR 4 in nvvmAddNVVMContainerToProgram, may need newer version of nvJitLink library
 

In [ ]:
d_out2 = cuda.device_array((28,28), dtype=np.float32)
launch_2d(conv2d, d_out1, kernels[1], d_out2)
launch_2d(relu, d_out2)

In [ ]:
d_pool1 = cuda.device_array((14,14), dtype=np.float32)
threads = (16,16)
blocks = (math.ceil(14/16), math.ceil(14/16))
maxpool_kernel[blocks, threads](d_out2, d_pool1)

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [ ]:
d_out3 = cuda.device_array((14,14), dtype=np.float32)
launch_2d(conv2d, d_pool1, kernels[2], d_out3)
launch_2d(relu, d_out3)

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [ ]:
d_out4 = cuda.device_array((14,14), dtype=np.float32)
launch_2d(conv2d, d_out3, kernels[3], d_out4)
launch_2d(relu, d_out4)

In [ ]:
d_pool2 = cuda.device_array((7,7), dtype=np.float32)
threads = (16,16)
blocks = (math.ceil(7/16), math.ceil(7/16))
maxpool_kernel[blocks, threads](d_out4, d_pool2)

In [ ]:
d_out5 = cuda.device_array((7,7), dtype=np.float32)
launch_2d(conv2d, d_pool2, kernels[4], d_out5)
launch_2d(relu, d_out5)

In [ ]:
final_output = d_out5.copy_to_host()
flattened = final_output.flatten()

print("Final feature size:", flattened.shape)

Final feature size: (49,)


In [ ]:
# fc_weights = np.random.randn(49, 10).astype(np.float32) * 0.01
fc_weights = np.random.randn(784, 10).astype(np.float32) * 0.01
fc_bias = np.zeros(10, dtype=np.float32)

d_fc_weights = cuda.to_device(fc_weights)
d_fc_bias = cuda.to_device(fc_bias)

In [ ]:
@cuda.jit
def fc_forward(x, weights, bias, output):
    i = cuda.grid(1)
    if i < 10:
        s = 0.0
        for j in range(x.shape[0]):
            s += x[j] * weights[j, i]
        output[i] = s + bias[i]

In [ ]:
@cuda.jit
def fc_forward(x, weights, bias, output):
    i = cuda.grid(1)
    if i < 10:
        s = 0.0
        for j in range(x.shape[0]):
            s += x[j] * weights[j, i]
        output[i] = s + bias[i]

In [ ]:
def softmax(x):
    e = np.exp(x - np.max(x))
    return e / np.sum(e)

def cross_entropy(pred, label):
    return -np.log(pred[label] + 1e-8)

In [ ]:
def fc_backward(x, pred, label, weights):
    grad_out = pred.copy()
    grad_out[label] -= 1.0   # dL/dz

    grad_w = np.outer(x, grad_out)
    grad_b = grad_out

    grad_input = np.dot(weights, grad_out)

    return grad_w, grad_b, grad_input

In [ ]:
learning_rate = 0.01

# fc_weights -= learning_rate * grad_w
# fc_bias -= learning_rate * grad_b

In [ ]:
# d_fc_weights = cuda.to_device(fc_weights)
# d_fc_bias = cuda.to_device(fc_bias)

In [ ]:
@cuda.jit
def conv2d_input_backward(grad_output, kernel, grad_input):
    h, w = cuda.grid(2)

    H = grad_input.shape[0]
    W = grad_input.shape[1]

    if h < H and w < W:
        s = 0.0

        for kh in range(3):
            for kw in range(3):
                oh = h - kh + 1
                ow = w - kw + 1

                if 0 <= oh < H and 0 <= ow < W:
                    s += grad_output[oh, ow] * kernel[kh, kw]

        grad_input[h, w] = s

In [ ]:
@cuda.jit
def conv2d_backward(input_img, grad_output, grad_kernel):
    kh, kw = cuda.grid(2)

    if kh < 3 and kw < 3:
        s = 0.0
        H = input_img.shape[0]
        W = input_img.shape[1]

        for h in range(H):
            for w in range(W):
                ih = h + kh - 1
                iw = w + kw - 1

                if 0 <= ih < H and 0 <= iw < W:
                    s += input_img[ih, iw] * grad_output[h, w]

        grad_kernel[kh, kw] = s

In [ ]:
@cuda.jit
def relu_backward(grad_output, activation, grad_input):
    h, w = cuda.grid(2)

    if h < activation.shape[0] and w < activation.shape[1]:
        if activation[h, w] > 0:
            grad_input[h, w] = grad_output[h, w]
        else:
            grad_input[h, w] = 0.0

In [ ]:
epochs = 5

In [ ]:
for epoch in range(epochs):
    total_loss = 0

    # for idx in range(len(train_dataset)):
    for idx in range(2000):

        image, label = train_dataset[idx]
        image = image.squeeze().numpy().astype(np.float32)

        # ---- Move to GPU ----
        d_img = cuda.to_device(image)

        # ---- Forward Conv Layers ----
        activations = []
        inputs = []

        current = d_img

        for k in range(5):

            inputs.append(current.copy_to_host())

            d_kernel = cuda.to_device(kernels[k])
            d_out = cuda.device_array(current.shape, dtype=np.float32)

            threads = (16,16)
            blocks = (math.ceil(current.shape[0]/16),
                      math.ceil(current.shape[1]/16))

            conv2d[blocks, threads](current, d_kernel, d_out)
            relu[blocks, threads](d_out)

            current = d_out
            activations.append(current.copy_to_host())

        # ---- Flatten ----
        final_feature = current.copy_to_host().flatten()

        # ---- FC Forward ----
        logits = np.dot(final_feature, fc_weights) + fc_bias
        pred = softmax(logits)

        loss = cross_entropy(pred, label)
        total_loss += loss

        # ---- FC Backward ----
        grad_w, grad_b, grad_input = fc_backward(
            final_feature, pred, label, fc_weights
        )

        # Update FC
        fc_weights -= learning_rate * grad_w
        fc_bias -= learning_rate * grad_b

        # ---- Backprop through Conv5 → Conv1 ----
        grad_input = grad_input.reshape(28,28)

        for k in reversed(range(5)):

            d_grad_output = cuda.to_device(grad_input)
            d_input = cuda.to_device(inputs[k])
            d_grad_kernel = cuda.device_array((3,3), dtype=np.float32)

            threads = (16,16)
            blocks = (1,1)

            conv2d_backward[blocks, threads](
                d_input, d_grad_output, d_grad_kernel
            )

            grad_kernel = d_grad_kernel.copy_to_host()

            kernels[k] -= learning_rate * grad_kernel

            d_grad_input = cuda.device_array(inputs[k].shape, dtype=np.float32)

            threads = (16,16)
            blocks = (math.ceil(inputs[k].shape[0]/16),
                      math.ceil(inputs[k].shape[1]/16))

            d_kernel_current = cuda.to_device(kernels[k])

            conv2d_input_backward[blocks, threads](
                d_grad_output, d_kernel_current, d_grad_input
            )

            grad_input = d_grad_input.copy_to_host()

    print("Epoch:", epoch+1, "Loss:", total_loss/len(train_dataset))

Epoch: 1 Loss: 0.07677122
Epoch: 2 Loss: 0.07677566
Epoch: 3 Loss: 0.07677639
Epoch: 4 Loss: 0.07677654
Epoch: 5 Loss: 0.07677651


In [ ]:
def predict(image):
    image = image.squeeze().numpy().astype(np.float32)
    d_img = cuda.to_device(image)

    current = d_img

    # 5 Conv Layers
    for k in range(5):
        d_kernel = cuda.to_device(kernels[k])
        d_out = cuda.device_array(current.shape, dtype=np.float32)

        threads = (16,16)
        blocks = (math.ceil(current.shape[0]/16),
                  math.ceil(current.shape[1]/16))

        conv2d[blocks, threads](current, d_kernel, d_out)
        relu[blocks, threads](d_out)

        current = d_out

    final_feature = current.copy_to_host().flatten()

    logits = np.dot(final_feature, fc_weights) + fc_bias
    pred = softmax(logits)

    return np.argmax(pred)

In [ ]:
image, label = test_dataset[0]

prediction = predict(image)

print("True Label:", label)
print("Predicted :", prediction)

True Label: 7
Predicted : 7
